# 项目分析目标
> 分析某电商平台用户购物行为路径，识别用户在点击→收藏→加购→购买各环节的转化特征与流失环节，为优化用户运营策略提供数据参考。

## 分析框架  
- 用户行为总览 — 整体行为分布  
- 转化漏斗分析 — 各环节转化率与流失点  
- 用户分层分析 — 不同购买力用户的特征  
- 时间维度分析 — 用户活跃的时间规律  

In [1]:
#导入数据库
import numpy as np
import pandas as pd

# 导入csv文件查看表格结构

In [2]:
#由于原始csv文件没有表头，因此需要手动添加表头
DATA_PATH = 'data/UserBehavior.csv'
columns = ['用户id' , '商品id', '商品类目id', '用户行为', '发生时间']
df = pd.read_csv(DATA_PATH, nrows=3000000, header= None, names = columns)

#查看表结构
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 5 columns):
 #   Column  Dtype 
---  ------  ----- 
 0   用户id    int64 
 1   商品id    int64 
 2   商品类目id  int64 
 3   用户行为    object
 4   发生时间    int64 
dtypes: int64(4), object(1)
memory usage: 114.4+ MB


# 数据清洗

In [3]:
#数据概览——前五行
print('\n列表前五行数据概览:')
print(df.head())

#缺失值查询
print('\n缺失值分布:')
print(df.isnull().sum())
print()

#重复值查询
print('\n重复值数量:')
print(df.duplicated().sum())
df = df.drop_duplicates()

#数据类型转换，将时间列的格式转换为datetime格式，注意时间戳！
df['发生时间'] = pd.to_datetime(df['发生时间'], unit='s')
df = df[df['发生时间'].between('2017-01-01','2017-12-31')]

#替换、修正文本
df['用户行为'] = df['用户行为'].replace({'pv': '点击', 'buy': '购买', 'cart': '加入购物车', 'fav': '收藏商品'})

#最终清洗结果统计
print('\n=====清洗后数据统计=====:')
print(f'清洗后的有效数据量:{df.shape[0]}')
print(f'数据覆盖时间范围:{df['发生时间'].min()}至{df['发生时间'].max()}')
print(f'数据覆盖用户数量:{df['用户id'].nunique()}')
print(f'数据覆盖商品数量：{df['商品id'].nunique()}')
print(f'数据覆盖商品种类{df['商品类目id'].nunique()}')
df


列表前五行数据概览:
   用户id     商品id   商品类目id 用户行为        发生时间
0     1  2268318  2520377   pv  1511544070
1     1  2333346  2520771   pv  1511561733
2     1  2576651   149192   pv  1511572885
3     1  3830808  4181361   pv  1511593493
4     1  4365585  2520377   pv  1511596146

缺失值分布:
用户id      0
商品id      0
商品类目id    0
用户行为      0
发生时间      0
dtype: int64


重复值数量:
1

=====清洗后数据统计=====:
清洗后的有效数据量:2999981
数据覆盖时间范围:2017-07-03 09:24:32至2017-12-06 12:06:46
数据覆盖用户数量:29178
数据覆盖商品数量：805067
数据覆盖商品种类6868


,用户id,商品id,商品类目id,用户行为,发生时间
0,1,2268318,2520377,点击,2017-11-24 17:21:10
1,1,2333346,2520771,点击,2017-11-24 22:15:33
2,1,2576651,149192,点击,2017-11-25 01:21:25
3,1,3830808,4181361,点击,2017-11-25 07:04:53
4,1,4365585,2520377,点击,2017-11-25 07:49:06
...,...,...,...,...,...
2999995,218275,2772813,1577687,点击,2017-11-29 15:10:10
2999996,218275,4654492,1577687,点击,2017-11-29 15:11:08
2999997,218275,2772813,1577687,点击,2017-11-29 15:11:17
2999998,218275,1207527,1577687,点击,2017-11-29 15:12:50


# 用户行为总览

In [4]:
custom_behavior_number = df.groupby('用户行为').size() #用户的行为构成
avg_behavior_number = (df['用户行为'].value_counts() / df['用户id'].nunique()).round() #人均行为数
behavior_ration = df['用户行为'].value_counts(normalize= True) #用户整体行为占比

#数据展示
print(f'用户各行为数量:{custom_behavior_number.sort_values(ascending= False)}')
print(f'\n用户人均行为数量:{avg_behavior_number.sort_values(ascending= False)}')
print(f'\n用户整体各行为占比:{behavior_ration.sort_values(ascending=False)}')
print()

#用户活跃度分析
user_activity = df.groupby('用户id').size() #每一位用户的行为数
user_activity_med = user_activity.median()
print(f'\n用户活跃次数中位数{user_activity_med}\n')

percentile = [1,10,20,30,40,50,60,70,80,90,99]
for p in percentile:
    value = user_activity.quantile(p/100)
    print(f'第{p}%分位数是:{value}') 

#寻找80%用户的活跃次数区间
p10 = user_activity.quantile(0.1)
p90 = user_activity.quantile(0.9)
print('\n====80%用户的行为分布====')
print(f'中间80%用户的行为量集中在分布在{p10}--{p90}次之间')

用户各行为数量:用户行为
点击       2685859
加入购物车     167572
收藏商品       86576
购买         59974
dtype: int64

用户人均行为数量:用户行为
点击       92.0
加入购物车     6.0
收藏商品      3.0
购买        2.0
Name: count, dtype: float64

用户整体各行为占比:用户行为
点击       0.895292
加入购物车    0.055858
收藏商品     0.028859
购买       0.019991
Name: proportion, dtype: float64


用户活跃次数中位数77.0

第1%分位数是:8.0
第10%分位数是:22.0
第20%分位数是:34.0
第30%分位数是:47.0
第40%分位数是:61.0
第50%分位数是:77.0
第60%分位数是:97.0
第70%分位数是:122.0
第80%分位数是:158.0
第90%分位数是:221.0
第99%分位数是:410.0

====80%用户的行为分布====
中间80%用户的行为量集中在分布在22.0--221.0次之间


# 转化漏斗分析  
## 1.转化率分析

In [5]:
#计算每个行为背后的用户数量
funnel_data = {}
for behavior in ['点击', '收藏商品', '加入购物车', '购买']:
    users = df[df['用户行为'] == behavior]['用户id'].nunique()
    funnel_data[behavior] = users
    print(f'{behavior}:{users}人')
print("""\n设定的原有转化路径是:点击-->收藏商品-->加入购物车-->购买
经过统计发现,收藏行为的人数过少,不符合漏斗分析层层递减的要求
这说明对许多用户而言,收藏不是必经之路,因此将收藏视为一种可选行为
更改后的转化路径为:点击-->加入购物车-->购买""")
print()

#定义新的漏斗路径
new_funnel_data = {}
for behavior in ['点击',  '加入购物车', '购买']:
    users = df[df['用户行为'] == behavior]['用户id'].nunique()
    new_funnel_data[behavior] = users
    print(f'{behavior}:{users}人')
print()
#计算各环节的环比转化率与整体转化率
print('-'*70)
print(f'{'行为环节':<12} {'用户数':10} {'环比转化率':<12} {'整体转化率':<12} ')
print('-'*70)

pre_users = None
clic_users = funnel_data['点击']
for behavior,users in new_funnel_data.items():
    if pre_users is None:
        #点击环节，没有环比转化率和流失率
        cvr = '-'
        overall = '100%'
        loss = '-'
    else:
        #环比转化率 = (本层用户数量/上一层用户数量)
        cvr_value = users/pre_users
        cvr = f'{cvr_value*100:.1f}%'
        #整体转化率 = (本层用户数量/起始层用户数量)
        overall_value = users/clic_users
        overall = f'{overall_value*100:.1f}%'
    print(f'{behavior:<12} {users:<15} {cvr:<15} {overall:<15} ')
    pre_users = users

点击:29049人
收藏商品:11701人
加入购物车:22061人
购买:19902人

设定的原有转化路径是:点击-->收藏商品-->加入购物车-->购买
经过统计发现,收藏行为的人数过少,不符合漏斗分析层层递减的要求
这说明对许多用户而言,收藏不是必经之路,因此将收藏视为一种可选行为
更改后的转化路径为:点击-->加入购物车-->购买

点击:29049人
加入购物车:22061人
购买:19902人

----------------------------------------------------------------------
行为环节         用户数        环比转化率        整体转化率        
----------------------------------------------------------------------
点击           29049           -               100%            
加入购物车        22061           75.9%           75.9%           
购买           19902           90.2%           68.5%           


## 2.流失率分析

In [6]:
print('-'*70)
print('流失分析')
print('-'*70)

stage = ['点击', '加入购物车', '购买']
for i in range(1, len(stage)):
    pre_stage = stage[i-1]
    cur_stage = stage[i]
    pre_count = new_funnel_data[pre_stage]
    cur_count = new_funnel_data[cur_stage]
    print(f'{pre_stage}-->{cur_stage}:流失{pre_count - cur_count}人, 流失率:{((pre_count - cur_count)/pre_count)*100:.1f}%')
print()

print("""流失率最严重的环节:点击-->加入购物车
流失率:24.1%
该环节流失了6988人
点击量高但是加入购物车的行为较少
说明在商品宣传环节上没有充分吸引用户,存在进一步优化的空间""")

----------------------------------------------------------------------
流失分析
----------------------------------------------------------------------
点击-->加入购物车:流失6988人, 流失率:24.1%
加入购物车-->购买:流失2159人, 流失率:9.8%

流失率最严重的环节:点击-->加入购物车
流失率:24.1%
该环节流失了6988人
点击量高但是加入购物车的行为较少
说明在商品宣传环节上没有充分吸引用户,存在进一步优化的空间


# 用户分层分析(RFM)

In [7]:
#在这里定义R:距离最近一次购买的天数； 
#         F:每个用户购买行为的次数； 
#         M:由于数据中没有用户的购买金额，所以在这里定义M为行为加权分数，用来衡量用户的活跃程度，其中:点击=1，收藏=2，加入购物车=2，购买=3，
rfm_analyse = df.copy()
rfm_analyse['发生日期'] = rfm_analyse['发生时间'].dt.normalize()
all_user = rfm_analyse['用户id'].unique()


#----------计算R、F、M----------
#1.计算R
purchas = rfm_analyse[rfm_analyse['用户行为'] == '购买']
reference_date = purchas['发生日期'].max() + pd.Timedelta(days=1)
R = (reference_date - purchas.groupby('用户id')['发生日期'].max()).dt.days
R_all_user_table = R.reindex(all_user,fill_value=999).reset_index().rename(columns = {'发生日期':'最近购买间隔_天'}) #在这里给从未有过购买行为的用户赋值999，代表其在短期内不会进行购买
print('====用户距离最近一次购买的天数====')
print(R_all_user_table)
print()

#2.计算F
F = rfm_analyse[rfm_analyse['用户行为'] == '购买'].groupby('用户id')['用户行为'].size()
F_all_user_table = F.reindex(all_user, fill_value=0).reset_index().rename(columns={'用户行为':'购买次数'}) #在这里给从未有过购买行为的用户赋值0，代表其购买次数为0
print('====用户购买次数====')
print(F_all_user_table)
print()

#3.计算M
#给各项行为进行赋值
score = {'点击':1 , '收藏商品':2 , '加入购物车':2, '购买':3}
rfm_analyse['行为得分'] = rfm_analyse['用户行为'].map(score)
M_table = rfm_analyse.groupby('用户id')['行为得分'].sum().reset_index().rename(columns={'行为得分':'行为加权得分'})
print('====用户行为得分====')
print(M_table)
print()

#4.合并R、F、M三张表
rfm =R_all_user_table.merge(F_all_user_table, on='用户id', how='inner').merge(M_table, on='用户id', how='inner')
print('\n====每个用户的R、F、M值====')
print(rfm)
print()

#----------给R、F、M进行打分，并据此进行用户分层----------
#1.给每位用户的R、F、M进行打分
#定义一个打分函数
def score_by_quantile(series, reverse=False):
    try:
        if reverse:
            bins = pd.qcut(series, q=5, labels=[5,4,3,2,1], duplicates='drop')
        else:
            bins = pd.qcut(series, q=5, labels=[1,2,3,4,5], duplicates='drop')
    except ValueError:
        if reverse:
            bins = pd.cut(series, bins=5, labels=[5,4,3,2,1])
        else:
            bins = pd.cut(series, bins=5, labels=[1,2,3,4,5])
    return bins.astype(int)

##应用函数进行打分
rfm['R_score'] = score_by_quantile(rfm['最近购买间隔_天'],reverse=True)
rfm['F_score'] = score_by_quantile(rfm['购买次数'], reverse=False)
rfm['M_score'] = score_by_quantile(rfm['行为加权得分'], reverse=False)
print('\n====打分完成后的rfm表====')
print(rfm)
print('\n====R、F、M的分布====')
print(rfm[['R_score', 'F_score', 'M_score']].describe())
##在这里观察数据发现，R值的分布两极分化严重，分数集中在1分和5分；而F的分数高度集中。
##因此应调整分箱策略，将R、F的值分成三箱而不是五箱； M的分数分布较为均匀，可以保持分成五箱不变
rfm['R_score'] = pd.qcut(rfm['最近购买间隔_天'], q=3, labels=[3,2,1], duplicates='drop').astype(int)
rfm['F_score'] = pd.qcut(rfm['购买次数'], q=3, labels=[1,2,3], duplicates='drop').astype(int)
print('\n====修改分箱策略后的R、F、M分布====')
print(rfm[['R_score', 'F_score', 'M_score']].describe())

#2.依据得分给用户进行分层
##R、F分别是用户最近一次购买的时间间隔和购买次数，用来衡量用户的购买力
##M是用户行为加权得分，用来衡量用户的活跃程度
rfm['total_score'] = rfm['R_score']*0.3 + rfm['F_score']*0.3 + rfm['M_score']*0.4 
##为了筛选出活跃高价值用户，对R（购买频率权重0.3）、F（购买力权重0.3）、M（活跃度权重0.4）进行加权合成综合得分，将用户分为A/B/C/D四级。
##查看加总综合分的分布，并据此制定分层条件
print('\n====综合分分布情况====')
print(rfm['total_score'].describe())
##加权综合得分的分布较为均匀，可以作为用户分层的依据

##依据综合得分进行分层
rfm['user_rate'] = pd.qcut(rfm['total_score'], q=4, labels=['D','C','B','A'])
print('\n====最终用户分层表====')
print(rfm[['用户id','user_rate']])

#3.统计各层信息
print('\n====每层的人数====')
print(rfm['user_rate'].value_counts().sort_values())



====用户距离最近一次购买的天数====
          用户id  最近购买间隔_天
0            1       999
1          100         6
2         1000       999
3      1000001         1
4      1000004       999
...        ...       ...
29173   218248         6
29174   218250       999
29175   218267         3
29176    21827       999
29177   218275         5

[29178 rows x 2 columns]

====用户购买次数====
          用户id  购买次数
0            1     0
1          100     8
2         1000     0
3      1000001     1
4      1000004     0
...        ...   ...
29173   218248     2
29174   218250     0
29175   218267     1
29176    21827     0
29177   218275     2

[29178 rows x 2 columns]

====用户行为得分====
          用户id  行为加权得分
0            1      55
1           13      15
2           19      76
3           21     343
4          100     120
...        ...     ...
29173  1017990      62
29174  1017994      41
29175  1017997     102
29176  1018000     189
29177  1018011      44

[29178 rows x 2 columns]


====每个用户的R、F、M值====
          用户id  最近

# 用户活跃时间维度分析

In [8]:
#用户11月份活跃度分析
nov = df.copy()
nov['发生日期'] = nov['发生时间'].dt.normalize()
nov = nov[nov['发生日期'].between('2017-11-01', '2017-11-30')]
#日活跃度分析
nov['发生日期'] = nov['发生时间'].dt.normalize()
daily_active_user = nov.groupby('发生日期')['用户id'].nunique()
print('====用户11月份活跃度分析====')
print(f'日均用户活跃{daily_active_user.mean().round()}')
print(f'最高单日活跃{daily_active_user.max()}')
print(f'最低单日活跃{daily_active_user.min()}')
print(f'最高单日活跃日期{daily_active_user.idxmax().date()}')
print(f'最低单日活跃日期{daily_active_user.idxmin().date()}')
print()

#周活跃度分析
nov['星期几'] = nov['发生时间'].dt.dayofweek
weekday_user = nov.groupby('星期几')['用户id'].nunique()
weekday_user.index = ['周一', '周二', '周三', '周四', '周五', '周六', '周日']
print('====周活跃度分布====')
print(weekday_user.sort_values(ascending=False))
print('''观察发现:除了周五外每日的活跃用户数量相差很小,因此建议在除了周五以外的时间进行推送活动。
此外依然要考虑:出现如此大的偏差会不会是数据出了问题。
本项目所采用的数据总计1亿条,为了项目分析上的方便只选取了前300万条数据进行分析,因此样本有可能会出现时序偏差''')

#小时范围活跃度分析
nov['小时'] = nov['发生时间'].dt.hour
hour_user = nov.groupby('小时')['用户id'].nunique()
print('\n====各时段活跃用户数====')
print(hour_user.sort_values(ascending= False))
print(f"""观察数据发现：
用户活跃早高峰:5:00--9:00,活跃人数:{hour_user[5:10].sum()}人
用户活跃午高峰:11:00--15:00,活跃人数:{hour_user[11:16].sum()}人
用户活跃晚高峰:22:00--4:00,活跃人数:{hour_user[hour_user.index >= 22].sum() + hour_user[hour_user.index < 4].sum()}人
建议在以上时段推送业务""")


====用户11月份活跃度分析====
日均用户活跃5540.0
最高单日活跃21766
最低单日活跃1
最高单日活跃日期2017-11-30
最低单日活跃日期2017-11-03

====周活跃度分布====
周四    21810
周三    21417
周日    21179
周二    21017
周一    20982
周六    20889
周五     5533
Name: 用户id, dtype: int64
观察发现:除了周五外每日的活跃用户数量相差很小,因此建议在除了周五以外的时间进行推送活动。
此外依然要考虑:出现如此大的偏差会不会是数据出了问题。
本项目所采用的数据总计1亿条,为了项目分析上的方便只选取了前300万条数据进行分析,因此样本有可能会出现时序偏差

====各时段活跃用户数====
小时
13    15749
12    15180
14    14863
11    14052
7     13632
5     13604
4     13319
8     13319
6     13205
3     13091
2     12993
9     12850
10    12781
1     11504
15    11390
0      9703
23     7939
16     7377
22     4308
17     3741
18     2196
21     1932
19     1535
20     1412
Name: 用户id, dtype: int64
观察数据发现：
用户活跃早高峰:5:00--9:00,活跃人数:66610人
用户活跃午高峰:11:00--15:00,活跃人数:71234人
用户活跃晚高峰:22:00--4:00,活跃人数:59538人
建议在以上时段推送业务


# 导出制表需要的数据

In [ ]:
#定义导出文件夹字段名
output_profile = 'tableau_data'

#漏斗数据
funnel_df = pd.DataFrame(list(new_funnel_data.items()),columns=['行为环节', '用户数'])
funnel_df['环比转化率'] = ['-', '75.9%', '90.2%']
funnel_df['整体转化率'] = ['100%', '75.9%', '68.5%']
funnel_df.to_csv(f'{output_profile}/01_漏斗数据.csv', index=False, encoding='utf-8-sig')


#行为类型占比数据
behavior_df = custom_behavior_number.reset_index()
behavior_df.columns = ['用户行为', '人数']
behavior_df.to_csv(f'{output_profile}/02_行为类型占比数据.csv', index=False, encoding='utf-8-sig')

#R、F、M分组数据
RFM_df = rfm['user_rate'].value_counts().reset_index()
RFM_df.columns = ['用户层级', '人数']
RFM_df.to_csv(f'{output_profile}/03_RFM用户分层.csv', index=False, encoding='utf-8-sig')

#小时活跃度数据
hour_df = hour_user.reset_index()
hour_df.columns = ['小时', '用户活跃数']
hour_df.to_csv(f'{output_profile}/04_各时段活跃用户.csv', index=False, encoding='utf-8-sig')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   行为环节    3 non-null      object
 1   用户数     3 non-null      int64 
 2   环比转化率   3 non-null      object
 3   整体转化率   3 non-null      object
dtypes: int64(1), object(3)
memory usage: 228.0+ bytes
